# บทที่ 5 การถดถอยและแบบจำลองที่มีข้อจำกัดทางฟิสิกส์
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch05_regression/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch05_regression/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch05_regression/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch05_regression'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'numpy': 'numpy', 'pandas': 'pandas', 'scipy': 'scipy', 'sklearn': 'scikit-learn>=1.3', 'statsmodels': 'statsmodels>=0.14'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_hooke_ols.py` — โค้ดที่ 5.1, 5.2, 5.3

Ch.5  Hooke's law by ordinary least squares: scikit-learn, statsmodels, and a pipeline.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_hooke_ols.py  จากโฟลเดอร์ของบท
__file__ = '01_hooke_ols.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.5  Hooke's law by ordinary least squares: scikit-learn, statsmodels, and a pipeline.

Book references: ssec:sklearn-hooke, ssec:coeff-se, ssec:sklearn-pipeline, ssec:err-prop
Listings reproduced line for line: code:hooke, code:statsmodels, code:pipeline
All data are SYNTHETIC (k_true = 5.0 N/m, noise 0.03 N, seed 2026).
"""
import os
from pathlib import Path

os.chdir(Path(__file__).resolve().parent)

### โค้ดที่ 5.1

In [ ]:
# --- code:hooke --------------------------------------------------------------
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(seed=2026)
k_true = 5.0                                   # N/m
x = np.linspace(0.0, 0.20, 50)                 # m
F_obs = k_true * x + rng.normal(scale=0.03, size=x.size)   # N

X = x.reshape(-1, 1)                           # shape (50, 1)
model = LinearRegression()
model.fit(X, F_obs)
print(f"k_fit = {model.coef_[0]:.4f} N/m, intercept = {model.intercept_:.4f} N")
print(f"R^2 = {model.score(X, F_obs):.4f}")

### โค้ดที่ 5.2

In [ ]:
# --- code:statsmodels --------------------------------------------------------
import statsmodels.api as sm

X_ic = sm.add_constant(X)             # add the column of ones for the intercept
ols_model = sm.OLS(F_obs, X_ic).fit()
print(ols_model.summary())            # coef, SE, t, p, 95% CI
print(ols_model.bse)                  # standard errors only
print(ols_model.conf_int(alpha=0.05)) # 95% confidence intervals

# --- numbers quoted in the text ---------------------------------------------
k_hat, se_k = ols_model.params[1], ols_model.bse[1]
lo, hi = ols_model.conf_int(alpha=0.05)[1]
print(f"\nk = {k_hat:.3f} +/- {se_k:.3f} N/m, 95 % CI [{lo:.2f}, {hi:.2f}] N/m")
print(f"k_true = 5.0 lies {(k_hat - k_true) / se_k:.1f} SE from the estimate "
      f"(outside the 95 % CI: {not lo <= k_true <= hi}); relative error {100 * (k_hat / k_true - 1):.1f} %")
cov = ols_model.cov_params()
print(f"correlation(intercept, slope) = {cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1]):.2f}")
print(f"sum of residuals = {ols_model.resid.sum():.1e} (zero when the model has an intercept)")

# how often does the 95 % CI miss k_true?  repeat the experiment with other seeds
miss = 0
for s in range(2000):
    r = np.random.default_rng(s)
    F = k_true * x + r.normal(scale=0.03, size=x.size)
    lo_s, hi_s = sm.OLS(F, X_ic).fit().conf_int(alpha=0.05)[1]
    miss += not lo_s <= k_true <= hi_s
print(f"2,000 repeated experiments: the 95 % CI misses k_true in {miss / 2000:.3f} of them")

### โค้ดที่ 5.3

In [ ]:
# --- code:pipeline -----------------------------------------------------------
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('poly',   PolynomialFeatures(degree=2, include_bias=False)),
    ('reg',    LinearRegression()),
])
pipe.fit(X, F_obs)          # X must be 2-D, as in the previous listing
F_pred = pipe.predict(X)
print(f"\npipeline (scale -> degree 2 -> OLS): MSE = {np.mean((F_pred - F_obs)**2):.2e} N^2")

## `02_bias_variance.py` — โค้ดที่ 5.4

Ch.5  Bias-variance trade-off: polynomial degree scan on an anharmonic spring.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_bias_variance.py  จากโฟลเดอร์ของบท
__file__ = '02_bias_variance.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.5  Bias-variance trade-off: polynomial degree scan on an anharmonic spring.

Book references: ssec:bias-variance, code:bv-scan (reproduced line for line)
SYNTHETIC data: F = k1 x + k3 x^3 + noise, k1 = 5 N/m, k3 = 500 N/m^3, sigma = 0.05 N, 60 points.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

os.chdir(Path(__file__).resolve().parent)
Path('data').mkdir(exist_ok=True); Path('outputs').mkdir(exist_ok=True)

### โค้ดที่ 5.4

In [ ]:
# --- code:bv-scan ------------------------------------------------------------
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(seed=2026)
k1, k3 = 5.0, 500.0                                  # N/m, N/m^3
x_an = rng.uniform(-0.15, 0.15, 60)
F_an = k1 * x_an + k3 * x_an**3 + rng.normal(scale=0.05, size=x_an.size)
X_an = x_an.reshape(-1, 1)

X_train, X_val, y_train, y_val = train_test_split(
    X_an, F_an, test_size=0.3, random_state=42)
mse_train, mse_val = [], []
for d in range(1, 16):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('poly',   PolynomialFeatures(degree=d, include_bias=False)),
        ('reg',    LinearRegression()),
    ])
    pipe.fit(X_train, y_train)
    mse_train.append(np.mean((pipe.predict(X_train) - y_train)**2))
    mse_val.append(np.mean((pipe.predict(X_val) - y_val)**2))

# --- report ------------------------------------------------------------------
pd.DataFrame({'x': x_an, 'F': F_an}).to_csv('data/anharmonic_spring.csv', index=False)
print(" d   MSE_train    MSE_val   (noise variance sigma^2 = 0.0025 N^2)")
for d, a, b in zip(range(1, 16), mse_train, mse_val):
    print(f"{d:2d}   {a:9.5f}  {b:9.5f}")
best = int(np.argmin(mse_val)) + 1
print(f"lowest validation MSE at degree {best}")
print(f"degrees 1-2: MSE between {min(mse_train[:2] + mse_val[:2]):.3f} and {max(mse_train[:2] + mse_val[:2]):.3f} N^2 "
      f"= {min(mse_train[:2] + mse_val[:2]) / 0.0025:.0f}-{max(mse_train[:2] + mse_val[:2]) / 0.0025:.0f} x sigma^2")

# degree-3 polynomial on ALL data, raw x (coefficients keep their units)
fit3 = sm.OLS(F_an, np.column_stack([np.ones_like(x_an), x_an, x_an**2, x_an**3])).fit()
th, se = fit3.params, fit3.bse
print(f"degree 3 on all data: theta1 = {th[1]:.2f} +/- {se[1]:.2f} N/m, "
      f"theta2 = {th[2]:.1f} +/- {se[2]:.1f} N/m^2, theta3 = {th[3]:.0f} +/- {se[3]:.0f} N/m^3")

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(range(1, 16), mse_train, 'o-', label='training')
ax.semilogy(range(1, 16), mse_val, 's-', label='validation')
ax.axhline(0.05**2, ls=':', color='k', label=r'$\sigma^2$')
ax.set_xlabel('polynomial degree'); ax.set_ylabel(r'MSE (N$^2$)'); ax.legend()
fig.tight_layout(); fig.savefig('outputs/ch05_bias_variance.png', dpi=150)
print("saved outputs/ch05_bias_variance.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `03_ridge_cv.py` — โค้ดที่ 5.5

Ch.5  Five-fold cross-validation to choose the ridge penalty for a degree-10 polynomial.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_ridge_cv.py  จากโฟลเดอร์ของบท
__file__ = '03_ridge_cv.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.5  Five-fold cross-validation to choose the ridge penalty for a degree-10 polynomial.

Book references: ssec:cv-basics, ssec:hyper-PI, code:cv (reproduced line for line)
Uses the anharmonic-spring data of code:bv-scan (same seed, regenerated here).
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

os.chdir(Path(__file__).resolve().parent)
Path('outputs').mkdir(exist_ok=True)

# data exactly as in code:bv-scan
rng = np.random.default_rng(seed=2026)
k1, k3 = 5.0, 500.0
x_an = rng.uniform(-0.15, 0.15, 60)
F_an = k1 * x_an + k3 * x_an**3 + rng.normal(scale=0.05, size=x_an.size)
X_an = x_an.reshape(-1, 1)

### โค้ดที่ 5.5

In [ ]:
# --- code:cv -----------------------------------------------------------------
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import Ridge

lambdas = np.logspace(-4, 2, 20)
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
cv_scores = []
for lam in lambdas:
    pipe = Pipeline([
        ('scale_x', StandardScaler()),
        ('poly',    PolynomialFeatures(degree=10, include_bias=False)),
        ('scale_f', StandardScaler()),
        ('reg',     Ridge(alpha=lam)),
    ])
    scores = cross_val_score(pipe, X_an, F_an, cv=kfold,
                             scoring='neg_mean_squared_error')
    cv_scores.append(-scores.mean())
optimal_lambda = lambdas[np.argmin(cv_scores)]

# --- report ------------------------------------------------------------------
print("  lambda     CV MSE")
for lam, s in zip(lambdas, cv_scores):
    print(f"{lam:9.4f}  {s:.5f}")
print(f"optimal lambda = {optimal_lambda:.3f}, CV MSE = {min(cv_scores):.4f} N^2 (sigma^2 = 0.0025)")
flat = lambdas[np.array(cv_scores) < 1.1 * min(cv_scores)]
print(f"lambdas within 10 % of the minimum: {flat.min():.3f} to {flat.max():.3f}  (a flat valley)")

# the left side of the curve is not smooth: look at the individual folds
from sklearn.linear_model import LinearRegression
print("\nper-fold CV MSE at small lambda (and lambda = 0, plain OLS of degree 10):")
for lam in (0.0, 1e-4, 4.3e-4, 0.3):
    reg = LinearRegression() if lam == 0 else Ridge(alpha=lam)
    p_ = Pipeline([('scale_x', StandardScaler()),
                   ('poly', PolynomialFeatures(degree=10, include_bias=False)),
                   ('scale_f', StandardScaler()), ('reg', reg)])
    s_ = -cross_val_score(p_, X_an, F_an, cv=kfold, scoring='neg_mean_squared_error')
    print(f"  lambda = {lam:<7g} folds {np.round(s_, 4)}  mean {s_.mean():.4f}")
tr, va = next(iter(kfold.split(X_an)))
print(f"fold 1: validation x reaches {x_an[va].max():.3f} m, training x only up to {x_an[tr].max():.3f} m "
      "-> the degree-10 polynomial must extrapolate there; this fold makes the bump near lambda ~ 4e-4")

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogx(lambdas, cv_scores, 'o-'); ax.axvline(optimal_lambda, ls=':')
ax.axhline(0.0025, ls='--', color='k', label=r'$\sigma^2$')
ax.set_xlabel(r'$\lambda$ (alpha)'); ax.set_ylabel(r'5-fold CV MSE (N$^2$)'); ax.legend()
fig.tight_layout(); fig.savefig('outputs/ch05_ridge_cv.png', dpi=150)
print("saved outputs/ch05_ridge_cv.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `04_damped_oscillator.py` — โค้ดที่ 5.6

Ch.5  Chi-squared fit of a damped oscillator with physical bounds.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_damped_oscillator.py  จากโฟลเดอร์ของบท
__file__ = '04_damped_oscillator.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.5  Chi-squared fit of a damped oscillator with physical bounds.

Book references: ssec:damped-fit, ssec:fitted-PI, ssec:chi2-red, code:damped-cf (line for line)
SYNTHETIC data: A = 1.0 m, gamma = 0.3 1/s, omega = 6.0 rad/s, phi = 0.5 rad, 200 points in 10 s,
noise 0.05 m, seed 2026.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

os.chdir(Path(__file__).resolve().parent)
Path('data').mkdir(exist_ok=True); Path('outputs').mkdir(exist_ok=True)

### โค้ดที่ 5.6

In [ ]:
# --- code:damped-cf ----------------------------------------------------------
from scipy.optimize import curve_fit

def damped_oscillator(t, A, gamma, omega, phi):
    return A * np.exp(-gamma * t) * np.cos(omega * t + phi)

rng = np.random.default_rng(seed=2026)
t = np.linspace(0, 10, 200)                               # s
sigma_x = 0.05                                            # m
x_obs = damped_oscillator(t, 1.0, 0.3, 6.0, 0.5) \
        + rng.normal(scale=sigma_x, size=t.size)

p0 = [1.0, 0.2, 6.0, 0.0]                                 # initial guess
bounds = ([0, 0, 0, -np.pi], [10, 5, 20, np.pi])          # physical limits
popt, pcov = curve_fit(damped_oscillator, t, x_obs,
                       p0=p0, bounds=bounds,
                       sigma=sigma_x * np.ones(t.size),
                       absolute_sigma=True)

A, gamma, omega, phi = popt
errs = np.sqrt(np.diag(pcov))
print(f"gamma = {gamma:.4f} +/- {errs[1]:.4f} 1/s")
print(f"omega = {omega:.4f} +/- {errs[2]:.4f} rad/s")

# --- numbers quoted in the text ---------------------------------------------
pd.DataFrame({'t_s': t, 'x_m': x_obs}).to_csv('data/damped_oscillator.csv', index=False)
print(f"\nA = {A:.4f} +/- {errs[0]:.4f} m, phi = {phi:.4f} +/- {errs[3]:.4f} rad")
print(f"omega is {(omega - 6.0) / errs[2]:+.1f} sigma from 6.0, gamma is {(gamma - 0.3) / errs[1]:+.1f} sigma from 0.3")

resid = x_obs - damped_oscillator(t, *popt)
chi2_min = np.sum((resid / sigma_x)**2)
nu = t.size - len(popt)
print(f"chi2 = {chi2_min:.1f}, nu = {nu}, chi2_red = {chi2_min / nu:.2f}, "
      f"sqrt(2/nu) = {np.sqrt(2 / nu):.2f}, p = {stats.chi2.sf(chi2_min, nu):.3f}")

omega0 = np.sqrt(omega**2 + gamma**2)
print(f"Q = omega0/(2 gamma) = {omega0 / (2 * gamma):.2f} (true {np.sqrt(36 + 0.09) / 0.6:.2f}); "
      f"1/gamma = {1 / gamma:.2f} s; oscillations in 1/gamma: {omega / gamma / (2 * np.pi):.1f}; "
      f"gamma/omega = {gamma / omega:.3f}; (omega0 - omega)/omega = {100 * (omega0 / omega - 1):.2f} %")

# multi-start: does the fit converge to the same minimum from other starting omegas?
same = []
for w0 in np.arange(1, 19):
    try:
        p, _ = curve_fit(damped_oscillator, t, x_obs, p0=[1.0, 0.2, w0, 0.0], bounds=bounds,
                         sigma=sigma_x * np.ones(t.size), absolute_sigma=True)
        same.append(abs(p[2] - omega) < 1e-3)
    except RuntimeError:
        same.append(False)
ok = np.arange(1, 19)[np.array(same)]
print(f"starting omega 1..18 rad/s: converged to omega = {omega:.3f} from {len(ok)} of 18 starts "
      f"({'all' if len(ok) == 18 else list(ok)})")

fig, (a1, a2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
a1.plot(t, x_obs, '.', ms=3, label='data'); a1.plot(t, damped_oscillator(t, *popt), label='fit')
a1.set_ylabel('x (m)'); a1.legend()
a2.plot(t, resid / sigma_x, '.', ms=3); a2.axhline(0, color='k', lw=0.8)
a2.set_xlabel('t (s)'); a2.set_ylabel(r'residual / $\sigma$')
fig.tight_layout(); fig.savefig('outputs/ch05_damped_fit.png', dpi=150)
print("saved outputs/ch05_damped_fit.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `05_shockley_diode.py` — โค้ดที่ 5.7

Ch.5  Weighted fit of the Shockley diode equation with physical bounds.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_shockley_diode.py  จากโฟลเดอร์ของบท
__file__ = '05_shockley_diode.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.5  Weighted fit of the Shockley diode equation with physical bounds.

Book references: ssec:shockley, code:shockley-cf (reproduced line for line)
SYNTHETIC data: I0 = 2 pA, n = 1.4, 36 voltages 0.30-0.65 V, 5 % proportional noise, seed 2026.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import curve_fit

os.chdir(Path(__file__).resolve().parent)
Path('outputs').mkdir(exist_ok=True)

### โค้ดที่ 5.7

In [ ]:
# --- code:shockley-cf --------------------------------------------------------
V_T = 0.02585                                   # V, thermal voltage at 300 K

def shockley(V, I0, n):
    return I0 * (np.exp(V / (n * V_T)) - 1)

rng = np.random.default_rng(seed=2026)
V = np.linspace(0.30, 0.65, 36)                 # V
I_obs = shockley(V, 2e-12, 1.4) * (1 + rng.normal(scale=0.05, size=V.size))

p0 = [1e-12, 1.5]
bounds = ([1e-15, 1.0], [1e-9, 2.0])            # I0 in A; n from device physics
popt_d, pcov_d = curve_fit(shockley, V, I_obs, p0=p0, bounds=bounds,
                           sigma=0.05 * I_obs, absolute_sigma=True)

# --- numbers quoted in the text ---------------------------------------------
I0, n = popt_d
e = np.sqrt(np.diag(pcov_d))
rho = pcov_d[0, 1] / (e[0] * e[1])
print(f"I0 = ({I0 * 1e12:.2f} +/- {e[0] * 1e12:.2f}) pA, n = {n:.3f} +/- {e[1]:.3f}")
print(f"correlation(I0, n) = {rho:.3f}")
print(f"V_T = k_B T / q at 300 K = {1.380649e-23 * 300 / 1.602176634e-19 * 1e3:.2f} mV")
print(f"current range: {I_obs.min():.2e} A to {I_obs.max():.2e} A")

# what the book warns about: unweighted fit gives the high-voltage points all the weight
pu, pcu = curve_fit(shockley, V, I_obs, p0=p0, bounds=bounds)
print(f"unweighted fit for comparison: I0 = {pu[0] * 1e12:.2f} pA, n = {pu[1]:.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.semilogy(V, I_obs, 'o', ms=4, label='data'); ax.semilogy(V, shockley(V, *popt_d), label='weighted fit')
ax.set_xlabel('V (V)'); ax.set_ylabel('I (A)'); ax.legend()
fig.tight_layout(); fig.savefig('outputs/ch05_shockley.png', dpi=150)
print("saved outputs/ch05_shockley.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `06_q_propagation.py` — โค้ดที่ 5.8

Ch.5  Error propagation with the full covariance matrix: quality factor Q and amplitude at 5 s.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_q_propagation.py  จากโฟลเดอร์ของบท
__file__ = '06_q_propagation.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.5  Error propagation with the full covariance matrix: quality factor Q and amplitude at 5 s.

Book references: ssec:err-prop, ex:Q-prop, code:q-prop (reproduced line for line)
Runs the fit of code:damped-cf first (taken from 04_damped_oscillator.py) to get popt, pcov, errs.
"""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
src = (HERE / '04_damped_oscillator.py').read_text()
exec(src[src.index('from scipy.optimize import curve_fit'):src.index('# --- numbers quoted in the text')])

### โค้ดที่ 5.8

In [ ]:
# --- code:q-prop -------------------------------------------------------------
Q = omega / (2 * gamma)
dQ_dom = 1 / (2 * gamma)
dQ_dga = -omega / (2 * gamma**2)

# naive: ignore the off-diagonal term
s_naive = np.sqrt(dQ_dom**2 * pcov[2, 2] + dQ_dga**2 * pcov[1, 1])

# full: include the covariance between gamma (index 1) and omega (index 2)
s_full = np.sqrt(dQ_dom**2 * pcov[2, 2] + dQ_dga**2 * pcov[1, 1]
                 + 2 * dQ_dom * dQ_dga * pcov[1, 2])

# matrix form, J ordered as (A, gamma, omega, phi)
J = np.array([0.0, dQ_dga, dQ_dom, 0.0])
s_matrix = np.sqrt(J @ pcov @ J)       # identical to s_full

rho = pcov / np.outer(errs, errs)      # correlation matrix

# --- numbers quoted in the text ---------------------------------------------
print(f"\nQ = {Q:.2f}; sigma_Q naive = {s_naive:.3f}, full = {s_full:.3f}, matrix = {s_matrix:.3f}")
print(f"Q (true, omega/(2 gamma) with 6.0 and 0.3) = {6.0 / 0.6:.2f}; distance {(Q - 10.0) / s_full:.1f} sigma")
np.set_printoptions(precision=2, suppress=True)
print("correlation matrix (A, gamma, omega, phi):")
print(rho)
print(f"rho(gamma, omega) = {rho[1, 2]:+.2f}, rho(A, gamma) = {rho[0, 1]:+.2f}, rho(omega, phi) = {rho[2, 3]:+.2f}")

# amplitude at t = 5 s: a = A exp(-5 gamma)
t5 = 5.0
a5 = A * np.exp(-gamma * t5)
Ja = np.array([np.exp(-gamma * t5), -t5 * A * np.exp(-gamma * t5), 0.0, 0.0])
s_a_naive = np.sqrt(np.sum(Ja**2 * np.diag(pcov)))
s_a_full = np.sqrt(Ja @ pcov @ Ja)
print(f"a(5 s) = {a5:.3f} m; sigma naive = {s_a_naive:.4f} m, full = {s_a_full:.4f} m "
      f"(naive / full = {s_a_naive / s_a_full:.2f})")

# check the linear propagation against a Monte Carlo draw from N(popt, pcov)
draws = np.random.default_rng(1).multivariate_normal(popt, pcov, size=200_000)
print(f"Monte Carlo from N(popt, pcov): sigma_Q = {np.std(draws[:, 2] / (2 * draws[:, 1])):.3f}, "
      f"sigma_a(5 s) = {np.std(draws[:, 0] * np.exp(-5 * draws[:, 1])):.4f} m")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E5_2_raw_vs_standardized.py`

E5.2  Slope from raw x and from standardized x: k_std = k_raw * s_x.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E5_2_raw_vs_standardized.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E5_2_raw_vs_standardized.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E5.2  Slope from raw x and from standardized x: k_std = k_raw * s_x."""
import os
from pathlib import Path

import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

os.chdir(Path(__file__).resolve().parent.parent)

rng = np.random.default_rng(seed=2026)                      # data of code:hooke
x = np.linspace(0.0, 0.20, 50)
F_obs = 5.0 * x + rng.normal(scale=0.03, size=x.size)
X = x.reshape(-1, 1)

k_raw = LinearRegression().fit(X, F_obs).coef_[0]
Z = StandardScaler().fit_transform(X)
k_std = LinearRegression().fit(Z, F_obs).coef_[0]
s_x = x.std()                                               # ddof = 0, as StandardScaler uses
print(f"raw slope          k_raw = {k_raw:.4f} N/m")
print(f"standardized slope k_std = {k_std:.4f} N   (units of F: z has no unit)")
print(f"k_raw * s_x (ddof=0)     = {k_raw * s_x:.4f} N   -> identical")
print(f"k_raw * s_x (ddof=1)     = {k_raw * x.std(ddof=1):.4f} N   -> differs slightly: StandardScaler divides by N")

# why scaling matters for a penalty: the same variable in metres and in millimetres
from sklearn.linear_model import Ridge
for unit, f in (('m', 1.0), ('mm', 1000.0)):
    c = Ridge(alpha=1.0).fit(X * f, F_obs).coef_[0] * f
    print(f"ridge (alpha = 1) with x in {unit:2s}: slope converted back to N/m = {c:.3f}")
print("-> the penalty depends on the unit of x unless the features are standardized first")

## `exercises/E5_3_degrees_1_3_6_10.py` — โค้ดที่ 5.4

E5.3  Polynomials of degree 1, 3, 6 and 10 fitted to the training part of code:bv-scan,

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E5_3_degrees_1_3_6_10.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E5_3_degrees_1_3_6_10.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E5.3  Polynomials of degree 1, 3, 6 and 10 fitted to the training part of code:bv-scan,
and how much each curve changes when the data are drawn again (variance)."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

os.chdir(Path(__file__).resolve().parent.parent)


def make(seed):
    rng = np.random.default_rng(seed=seed)
    x = rng.uniform(-0.15, 0.15, 60)
    F = 5.0 * x + 500.0 * x**3 + rng.normal(scale=0.05, size=x.size)
    return train_test_split(x.reshape(-1, 1), F, test_size=0.3, random_state=42)


def model(d):
    return Pipeline([('scaler', StandardScaler()),
                     ('poly', PolynomialFeatures(degree=d, include_bias=False)),
                     ('reg', LinearRegression())])


X_train, X_val, y_train, y_val = make(2026)
xg = np.linspace(-0.15, 0.15, 400).reshape(-1, 1)
truth = 5.0 * xg[:, 0] + 500.0 * xg[:, 0]**3
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(X_train, y_train, 'k.', label='training data')
ax.plot(xg, truth, 'k:', lw=1, label='true law')
for d in (1, 3, 6, 10):
    m = model(d).fit(X_train, y_train)
    ax.plot(xg, m.predict(xg), label=f'degree {d}')
    r = y_train - m.predict(X_train)
    # residual sign runs: a systematic pattern gives few long runs
    runs = 1 + np.sum(np.diff(np.sign(r[np.argsort(X_train[:, 0])])) != 0)
    print(f"degree {d:2d}: largest |curve - true law| on the grid {np.max(np.abs(m.predict(xg) - truth)):.3f} N, "
          f"residual sign runs {runs} of {len(r)} points")
ax.set_ylim(-3, 3); ax.set_xlabel('x (m)'); ax.set_ylabel('F (N)'); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig('outputs/E5_3_degrees.png', dpi=150)

# variance: redraw the data 200 times and look at the spread of each curve
print("\nspread (std) of the fitted curve over 200 new data sets, at the centre and near the edge:")
edge = np.array([[0.145]]); centre = np.array([[0.0]])
for d in (1, 3, 6, 10):
    pe, pc = [], []
    for s in range(200):
        Xt, _, yt, _ = make(10_000 + s)
        m = model(d).fit(Xt, yt)
        pe.append(m.predict(edge)[0]); pc.append(m.predict(centre)[0])
    print(f"degree {d:2d}: x = 0: {np.std(pc):.3f} N   x = 0.145 m: {np.std(pe):.3f} N")
print("saved outputs/E5_3_degrees.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E5_4_lambda_vs_noise.py` — โค้ดที่ 5.5

E5.4  Optimal ridge lambda (code:cv) for noise 0.05 N and 0.15 N.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E5_4_lambda_vs_noise.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E5_4_lambda_vs_noise.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E5.4  Optimal ridge lambda (code:cv) for noise 0.05 N and 0.15 N."""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

os.chdir(Path(__file__).resolve().parent.parent)
lambdas = np.logspace(-4, 2, 20)
kfold = KFold(n_splits=5, shuffle=True, random_state=0)

fig, ax = plt.subplots(figsize=(6, 4))
for noise in (0.05, 0.15):
    rng = np.random.default_rng(seed=2026)                 # same draws as code:bv-scan
    x_an = rng.uniform(-0.15, 0.15, 60)
    F_an = 5.0 * x_an + 500.0 * x_an**3 + rng.normal(scale=noise, size=x_an.size)
    X_an = x_an.reshape(-1, 1)
    cv = []
    for lam in lambdas:
        pipe = Pipeline([('scale_x', StandardScaler()),
                         ('poly', PolynomialFeatures(degree=10, include_bias=False)),
                         ('scale_f', StandardScaler()),
                         ('reg', Ridge(alpha=lam))])
        cv.append(-cross_val_score(pipe, X_an, F_an, cv=kfold, scoring='neg_mean_squared_error').mean())
    cv = np.array(cv); k = np.argmin(cv)
    flat = lambdas[cv < 1.1 * cv[k]]
    print(f"noise {noise:.2f} N: lambda* = {lambdas[k]:.3f}, min CV MSE = {cv[k]:.4f} N^2 "
          f"(sigma^2 = {noise**2:.4f}); within 10 % of the minimum: {flat.min():.3f}-{flat.max():.3f}")
    ax.loglog(lambdas, cv, 'o-', label=f'noise {noise} N')
ax.set_xlabel(r'$\lambda$'); ax.set_ylabel('5-fold CV MSE'); ax.legend()
fig.tight_layout(); fig.savefig('outputs/E5_4_lambda_vs_noise.png', dpi=150)
print("saved outputs/E5_4_lambda_vs_noise.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E5_5_damped_from_graph.py`

E5.5  Starting values read from the graph, the fit, Q, and the residuals of the damped oscillator.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E5_5_damped_from_graph.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E5_5_damped_from_graph.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E5.5  Starting values read from the graph, the fit, Q, and the residuals of the damped oscillator."""
import os
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.signal import find_peaks

os.chdir(Path(__file__).resolve().parent.parent)
if not Path('data/damped_oscillator.csv').exists():
    os.system('python 04_damped_oscillator.py > /dev/null')
d = pd.read_csv('data/damped_oscillator.csv')
t, x = d['t_s'].to_numpy(), d['x_m'].to_numpy()


def damped_oscillator(t, A, gamma, omega, phi):
    return A * np.exp(-gamma * t) * np.cos(omega * t + phi)


# 1. starting values from the graph
pk, _ = find_peaks(x, distance=15)                # maxima at least ~0.75 s apart
cycles = len(pk) - 1
w_guess = 2 * np.pi * cycles / (t[pk[-1]] - t[pk[0]])
A_guess = x[pk[0]]
g_guess = np.log(x[pk[0]] / x[pk[4]]) / (t[pk[4]] - t[pk[0]])
print(f"{len(pk)} maxima; omega from their spacing = {w_guess:.2f} rad/s; "
      f"A from the first maximum = {A_guess:.2f} m; gamma from maxima 1 and 5 = {g_guess:.2f} 1/s")

# 2. fit
popt, pcov = curve_fit(damped_oscillator, t, x, p0=[A_guess, g_guess, w_guess, 0.0],
                       bounds=([0, 0, 0, -np.pi], [10, 5, 20, np.pi]),
                       sigma=0.05 * np.ones(t.size), absolute_sigma=True)
A, g, w, phi = popt
e = np.sqrt(np.diag(pcov))
print(f"gamma = ({g:.3f} +/- {e[1]:.3f}) 1/s, omega = ({w:.3f} +/- {e[2]:.3f}) rad/s")
print(f"relative uncertainty: gamma {100 * e[1] / g:.1f} %, omega {100 * e[2] / w:.2f} %")

# 3. Q with full propagation
J = np.array([0.0, -w / (2 * g**2), 1 / (2 * g), 0.0])
Q, sQ = w / (2 * g), np.sqrt(J @ pcov @ J)
Q_true = np.sqrt(6.0**2 + 0.3**2) / (2 * 0.3)
print(f"Q = {Q:.1f} +/- {sQ:.1f}; true Q = omega0/(2 gamma) = {Q_true:.2f}; "
      f"difference {(Q - Q_true) / sQ:.1f} sigma")

# 4. residuals
r = x - damped_oscillator(t, *popt)
print(f"residuals: mean {r.mean():+.4f} m, std {r.std(ddof=4):.4f} m (noise 0.05 m); "
      f"lag-1 autocorrelation {np.corrcoef(r[:-1], r[1:])[0, 1]:+.2f}")

## `exercises/E5_6_diode_series_resistance.py`

E5.6  Residual analysis: an ideal Shockley fit to a diode that has a series resistance.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E5_6_diode_series_resistance.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E5_6_diode_series_resistance.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E5.6  Residual analysis: an ideal Shockley fit to a diode that has a series resistance.

SYNTHETIC data: I0 = 2 pA, n = 1.4, R_s = 20 ohm, 46 voltages 0.30-0.75 V (step 0.01 V),
5 % proportional noise, seed 2026.  The implicit equation I = I0 {exp[(V - I R_s)/(n V_T)] - 1}
is solved with the Lambert W function.
"""
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
from scipy.special import lambertw

os.chdir(Path(__file__).resolve().parent.parent)
Path('data').mkdir(exist_ok=True)
V_T = 0.02585


def shockley(V, I0, n):
    return I0 * (np.exp(V / (n * V_T)) - 1)


def shockley_rs(V, I0, n, Rs):
    """Exact solution of I = I0 (exp((V - I Rs)/(n V_T)) - 1) via Lambert W."""
    a = n * V_T
    arg = I0 * Rs / a * np.exp((V + I0 * Rs) / a)
    return a / Rs * np.real(lambertw(arg)) - I0


rng = np.random.default_rng(seed=2026)
V = np.linspace(0.30, 0.75, 46)
I_true = shockley_rs(V, 2e-12, 1.4, 20.0)
I_obs = I_true * (1 + rng.normal(scale=0.05, size=V.size))
pd.DataFrame({'V': V, 'I_A': I_obs}).to_csv('data/diode_series_R.csv', index=False)
print(f"check: solution satisfies the implicit equation to "
      f"{np.max(np.abs(I_true - shockley(V - I_true * 20.0, 2e-12, 1.4)) / I_true):.1e} (relative)")

# ideal model, as in code:shockley-cf
sig = 0.05 * I_obs
p, pc = curve_fit(shockley, V, I_obs, p0=[1e-12, 1.5], bounds=([1e-15, 1.0], [1e-9, 2.0]),
                  sigma=sig, absolute_sigma=True)
z = (I_obs - shockley(V, *p)) / sig
nu = V.size - 2
print(f"\nideal Shockley: I0 = {p[0] * 1e12:.2f} pA, n = {p[1]:.3f}")
print(f"chi2_red = {np.sum(z**2) / nu:.1f} at nu = {nu}; sqrt(2/nu) = {np.sqrt(2 / nu):.2f}; "
      f"(chi2_red - 1)/sqrt(2/nu) = {(np.sum(z**2) / nu - 1) / np.sqrt(2 / nu):.0f}")
thirds = np.array_split(np.arange(V.size), 3)
print("mean normalised residual in the low / middle / high third of the voltage range: "
      + " / ".join(f"{z[i].mean():+.1f}" for i in thirds))
print(f"most negative normalised residual {z.min():.1f} sigma at V = {V[np.argmin(z)]:.2f} V "
      f"(last point: {z[-1]:.1f} sigma)")

# model with series resistance: three parameters
p3, pc3 = curve_fit(shockley_rs, V, I_obs, p0=[1e-12, 1.5, 10.0],
                    bounds=([1e-15, 1.0, 0.0], [1e-9, 2.0, 1e3]), sigma=sig, absolute_sigma=True)
e3 = np.sqrt(np.diag(pc3))
z3 = (I_obs - shockley_rs(V, *p3)) / sig
print(f"\nwith R_s: I0 = {p3[0] * 1e12:.2f} +/- {e3[0] * 1e12:.2f} pA, n = {p3[1]:.3f} +/- {e3[1]:.3f}, "
      f"R_s = {p3[2]:.1f} +/- {e3[2]:.1f} ohm; chi2_red = {np.sum(z3**2) / (V.size - 3):.2f}")

# only the low-voltage part: is R_s visible there?
low = V <= 0.55
pl, _ = curve_fit(shockley, V[low], I_obs[low], p0=[1e-12, 1.5], bounds=([1e-15, 1.0], [1e-9, 2.0]),
                  sigma=sig[low], absolute_sigma=True)
zl = (I_obs[low] - shockley(V[low], *pl)) / sig[low]
print(f"ideal fit on V <= 0.55 V only: n = {pl[1]:.3f}, chi2_red = {np.sum(zl**2) / (low.sum() - 2):.2f} "
      f"-> R_s is invisible at low current")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(V, z, 'o', ms=4, label='ideal Shockley'); ax.plot(V, z3, 's', ms=3, label='with $R_s$')
ax.axhline(0, color='k', lw=0.8); ax.set_xlabel('V (V)'); ax.set_ylabel(r'$(I_{obs}-\hat I)/\sigma$'); ax.legend()
fig.tight_layout(); fig.savefig('outputs/E5_6_diode_residuals.png', dpi=150)
print("saved outputs/E5_6_diode_residuals.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E5_7_correlated_quantities.py`

E5.7  When does ignoring correlations matter?  Q (no), amplitude at 5 s (yes, A-gamma),

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E5_7_correlated_quantities.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E5_7_correlated_quantities.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E5.7  When does ignoring correlations matter?  Q (no), amplitude at 5 s (yes, A-gamma),
and the time of a late crest of the cosine factor, t_k = (2 pi k - phi)/omega (yes, omega-phi)."""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent.parent
os.chdir(HERE)
src = (HERE / '04_damped_oscillator.py').read_text()
exec(src[src.index('from scipy.optimize import curve_fit'):src.index('# --- numbers quoted in the text')])


def compare(name, value, J):
    naive = np.sqrt(np.sum(J**2 * np.diag(pcov)))
    full = np.sqrt(J @ pcov @ J)
    print(f"{name:28s} = {value:8.4f}: naive {naive:.4f}, full {full:.4f}, naive/full = {naive / full:.2f}")


print()
compare("Q = omega/(2 gamma)", omega / (2 * gamma), np.array([0, -omega / (2 * gamma**2), 1 / (2 * gamma), 0]))
compare("a(5 s) = A exp(-5 gamma) [m]", A * np.exp(-5 * gamma),
        np.array([np.exp(-5 * gamma), -5 * A * np.exp(-5 * gamma), 0, 0]))
for k in (1, 9):
    tk = (2 * np.pi * k - phi) / omega
    compare(f"crest time t_{k} [s]", tk, np.array([0, 0, -tk / omega, -1 / omega]))
print("-> Q: gamma and omega are almost uncorrelated, both formulas agree.\n"
      "   a(5 s) and crest times: the derivatives have the same sign and rho < 0 (omega-phi) or\n"
      "   opposite signs and rho > 0 (A-gamma), so the covariance term is negative and the naive\n"
      "   formula overestimates the uncertainty (table tab:correlation-sig)")